# 🪜 Лестница масштабирования TinyGPT v3: формулы vs галлюцинации

Исследовательские вопросы:
1. **Снижает ли обучение на ПРАВИЛЬНЫХ формулах репозитория количество галлюцинаций?**
2. Как эффект меняется с ростом числа параметров (S 447K → M ~1.1M → L 4.15M → XL ~12M)?

Дизайн A/B: рука `formula` (истинный корпус) против руки `control` (та же грамматика,
но численные результаты в парах детерминированно искажены — сид 42) против `untrained`.
Метрика — **confident_hallucination_rate**: доля оформленных ответов с неверными числами
(«Ловушка Полезности» из монографии проекта).

> ⚠️ TinyGPT v3 — NumPy-only (ADR-001): **GPU не ускоряет**. Colab полезен быстрым CPU,
> долгими сессиями и отсутствием помех. Рекомендуемый runtime: **CPU** (или T4 — не повредит).

Прерывание безопасно: веса и история сохраняются каждую эпоху, повторный запуск
ячейки продолжит обучение с места остановки (resume).

**Требование:** в репозитории на GitHub уже должен быть `research/tinygpt_formula/`
со папкой `scaling/` (есть в ZIP-пакете rmt-llm-push после пуша через Termux-скрипт).

In [ ]:
# 1. Клонируем репозиторий
REPO_URL = 'https://github.com/wild8highlander/rmt-llm-research.git'
!git clone --depth 1 {REPO_URL} 2>/dev/null || echo 'репозиторий уже склонирован'
import os
os.chdir('/content/rmt-llm-research/research/tinygpt_formula')
print('CWD:', os.getcwd())
assert os.path.isdir('scaling') and os.path.isdir('scripts'), 'нет папки scaling/ — запушьте обновлённый репозиторий'

In [ ]:
# 2. Зависимости: только NumPy (+ matplotlib для графика; в Colab уже есть)
import numpy, matplotlib
print('numpy', numpy.__version__, '| matplotlib', matplotlib.__version__)

In [ ]:
# 3. Контрольный корпус (детерминирован, сид 42)
!python3 scaling/build_control_corpus.py

In [ ]:
# 4. Быстрая проверка пайплайна (2 эпохи на размер S, ~2 мин)
%env SCALING_SMOKE=1
%env SCALING_CELLS=S_formula
%env SCALING_MAX_MINUTES=6
!python3 scaling/scaling_experiment.py
# после проверки удаляем smoke-результаты, чтобы не попали в сводку
!rm -rf scaling/runs

In [ ]:
# 5. ОСНОВНОЙ ЗАПУСК: размеры S и M, обе руки (~1.5-2.5 ч суммарно)
# Если сессия прервётся — просто запустите ячейку ещё раз: продолжит с места остановки.
%env SCALING_CELLS=S_formula,S_control,M_formula,M_control
%env SCALING_MAX_MINUTES=240
!python3 scaling/scaling_experiment.py

### Опционально: размеры L (~4.15M) и XL (~12M)
L — это `config_train_4m` из основного ноутбука. На CPU Colab это ~2-4 ч на руку;
запускайте в отдельной сессии (или на Kaggle: 30 ч/нед бесплатно).

In [ ]:
# 6. Опционально: L и XL (запускайте, если готовы ждать несколько часов)
RUN_LARGE = False  # поменяйте на True для запуска
if RUN_LARGE:
    %env SCALING_CELLS=L_formula,L_control,XL_formula,XL_control
    %env SCALING_MAX_MINUTES=540
    !python3 scaling/scaling_experiment.py
else:
    print('пропущено (RUN_LARGE = False)')

In [ ]:
# 7. Результаты: таблицы, сводка, график
from IPython.display import Markdown, display, Image
try:
    display(Markdown(open('scaling/SCALING_RESULTS.md').read()))
except FileNotFoundError:
    print('сначала запустите ячейку 5')
try:
    display(Image(filename='scaling/hallucination_vs_params.png'))
except FileNotFoundError:
    pass

In [ ]:
# 8. Сохранить артефакты в Google Drive (по желанию)
SAVE_TO_DRIVE = False
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    !cp -r scaling/SCALING_RESULTS.md scaling/scaling_report.json scaling/hallucination_vs_params.png \
           /content/drive/MyDrive/ 2>/dev/null && echo 'сохранено в Drive'
else:
    from google.colab import files
    # files.download('scaling/SCALING_RESULTS.md')  # раскомментируйте для скачивания
    print('артефакты лежат в /content/rmt-llm-research/research/tinygpt_formula/scaling/')

## Как читать результаты

- **HELD-OUT** — параметры, которых не было в корпусе: уровень экстраполяции.
- **IN-DISTRIBUTION** — пары из корпуса: у модели есть знание. Здесь видно,
  воспроизводит ли она истину (formula) или выученную ложь (control).
- На S (447K) numeric-точность упирается в ёмкость; ожидается, что разрыв рук
  по `confident_hallucination_rate` и `numeric_score` будет расти к M/L/XL.
- Все числа пишутся в `scaling/SCALING_RESULTS.md` и `scaling_report.json`;
  метрики обоих бенчмарков считаются против ИСТИННЫХ значений библиотеки.